In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ z= xW + b $$
$$ p = \frac{1}{1+e^{-z}} $$
$$ L = -y \log(p) - (1-y) \log(1-p) $$

$$ \diamonds \diamonds \diamonds $$

$$ \frac{\partial L}{\partial x} = \frac{\partial L}{\partial z} \frac{\partial z}{\partial x} = (p-y) \cdot W^T $$
$$ \frac{\partial L}{\partial W} = \frac{\partial L}{\partial z} \frac{\partial z}{\partial W} = x^T(p-y) $$
$$ \frac{\partial L}{\partial b} = \frac{\partial L}{\partial z} \frac{\partial z}{\partial b} = p-y $$


In [ ]:
import torch as tr
import torch.nn as nn
import torch.optim as optim
import torch.autograd as autograd

import import_ipynb
import sigmoid # type: ignore
import linear # type: ignore
import binary_cross_entropy # type: ignore
from common import assert_eq, average_run, test_perceptron_boolean # type: ignore


class Per_Lin_Sig_BCE_Gradient_Function(autograd.Function):
    @staticmethod
    def forward(ctx, x, W, b, y):
        z = linear.linear(x, W, b)
        p = sigmoid.sigmoid(z)
        L = binary_cross_entropy.binary_cross_entropy(p, y)

        ctx.save_for_backward(x, W, b, y, p)

        return L
    
    @staticmethod
    def backward(ctx, dF_dL):
        (x, W, b, y, p) = ctx.saved_tensors

        S = x.shape[0]  # Samples
        FI = x.shape[1] # Features In
        FO = W.shape[1] # Features Out
        
        assert_eq(x.shape, (S, FI))
        assert_eq(W.shape, (FI, FO))
        assert_eq(b.shape, (FO,))
        assert_eq(y.shape, (S, FO))
        assert_eq(p.shape, (S, FO))

        # Average over samples
        dL_dz = (p - y) / S

        # (S, FI) = (S, FO) @ (FI, FO).T
        dL_dx = dL_dz @ W.t()
        assert_eq(dL_dx.shape, (S, FI))
        
        # (FI, FO) = (S, FI).T @ (S, FO)
        dL_dW = x.t() @ dL_dz
        assert_eq(dL_dW.shape, (FI, FO))

        # We need to calculete mean for dL_db as well,
        # however dL_dz is already averaged over samples,
        # so we can just sum over samples.
        dL_db = dL_dz.sum(dim=0)
        assert_eq(dL_db.shape, (FO,))

        return (dF_dL * dL_dx, dF_dL * dL_dW, dF_dL * dL_db, None)
    

class Per_Lin_Sig_BCE_Gradient(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.weight = nn.Parameter(tr.randn(out_features, in_features, dtype=tr.float32))
        self.bias = nn.Parameter(tr.randn(out_features, dtype=tr.float32))

    def forward(self, x, y):
        return Per_Lin_Sig_BCE_Gradient_Function.apply(x, self.weight.T, self.bias, y)
    
    # Extension
    # To facilitate testing and experimentation various perceptrons.
    def learn(self, x, y):
        return self(x, y)

    # Extension
    # To facilitate testing and experimentation various perceptrons.
    def predict(self, x):
        with tr.no_grad():
            return sigmoid.SigmoidFunction.apply(linear.LinearFunction.apply(x, self.weight.T, self.bias)) >= 0.5


def test_Per_Lin_Sig_BCE_Gradient(bool_fn, epochs, lr=0.1):
    return test_perceptron_boolean(Per_Lin_Sig_BCE_Gradient(2, 1), bool_fn, epochs=epochs, lr=lr)


if __name__ == "__main__":
    N=10

    logical_and = tr.logical_and
    print(f" AND, {5*N} epochs: {average_run(1*N, lambda: test_Per_Lin_Sig_BCE_Gradient(logical_and, epochs=5*N)):.2f}")

    logical_or = tr.logical_or
    print(f"  OR, {5*N} epochs: {average_run(1*N, lambda: test_Per_Lin_Sig_BCE_Gradient(logical_or, epochs=5*N)):.2f}")

    logical_nand = lambda x1, x2: tr.logical_not(tr.logical_and(x1, x2))
    print(f"NAND, {5*N} epochs: {average_run(1*N, lambda: test_Per_Lin_Sig_BCE_Gradient(logical_nand, epochs=5*N)):.2f}")

    logical_xor = lambda x1, x2: tr.logical_xor(x1, x2)
    print(f" XOR, {5*N} epochs: {average_run(1*N, lambda: test_Per_Lin_Sig_BCE_Gradient(logical_xor, epochs=5*N)):.2f}")


 AND, 500 epochs: 1.00


KeyboardInterrupt: 